# Day 25 — Project: KPI Dashboard from BW Extract
**Estimated time:** 90 minutes  
**Learning objectives:**
1. Work with a BW-style flat extract to derive YoY comparisons and trend KPIs
2. Build a customer retention flag and returns analysis
3. Produce 5 KPI cards (text output) and 3 analytical charts

---
> **SAP Context:** BW InfoCubes and DSOs are often extracted to flat CSV files for downstream analytics. The BW extract here mirrors what you'd get from a BEx query or an Open Hub destination: one row per KUNNR/MATNR/VKORG/CAL_YEAR_MONTH combination with pre-aggregated KPI values. CAL_YEAR_MONTH is the standard BW time characteristic (YYYYMM format).


## 0. Setup & Data Load

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path
from textwrap import dedent

pd.set_option('display.float_format', '{:,.2f}'.format)
plt.style.use('seaborn-v0_8-whitegrid')

from analytics_bootcamp.config import RAW_DATA_DIR as DATA
bw = pd.read_csv(DATA / 'bw_sales_kpis.csv')
bw['YEAR'] = (bw['CAL_YEAR_MONTH'] // 100).astype(int)
bw['MONTH'] = (bw['CAL_YEAR_MONTH'] % 100).astype(int)
bw['DATE'] = pd.to_datetime(bw['CAL_YEAR_MONTH'].astype(str), format='%Y%m')

print('Shape:', bw.shape)
print('Years:', sorted(bw['YEAR'].unique()))
bw.head(3)

## 1. Revenue vs Prior Year (YoY)

In [ ]:
# YOUR SOLUTION
# Aggregate revenue by year and month
monthly_rev = (
    bw.groupby(['YEAR', 'MONTH'])['REVENUE']
    .sum()
    .reset_index()
    .sort_values(['YEAR', 'MONTH'])
)

# Pivot to get current year vs prior year side by side
years = sorted(bw['YEAR'].unique())
CURRENT_YEAR = max(years)
PRIOR_YEAR = CURRENT_YEAR - 1

cy = monthly_rev[monthly_rev['YEAR'] == CURRENT_YEAR][['MONTH','REVENUE']].rename(columns={'REVENUE':'CY_REVENUE'})
py = monthly_rev[monthly_rev['YEAR'] == PRIOR_YEAR][['MONTH','REVENUE']].rename(columns={'REVENUE':'PY_REVENUE'})

yoy = cy.merge(py, on='MONTH', how='left')
yoy['YOY_CHANGE'] = yoy['CY_REVENUE'] - yoy['PY_REVENUE']
yoy['YOY_PCT'] = (yoy['YOY_CHANGE'] / yoy['PY_REVENUE'] * 100).round(1)

total_cy = yoy['CY_REVENUE'].sum()
total_py = yoy['PY_REVENUE'].sum()
yoy_total_pct = (total_cy - total_py) / total_py * 100

print(yoy.to_string(index=False))
print(f'\nFull-Year YoY: {yoy_total_pct:+.1f}%')

## 2. Gross Margin % Trend by Region

In [ ]:
# YOUR SOLUTION
gm_region = (
    bw.groupby(['YEAR', 'MONTH', 'REGION'])
    .agg(REVENUE=('REVENUE','sum'), COGS=('COGS','sum'), GM=('GROSS_MARGIN','sum'))
    .assign(GM_PCT=lambda d: (d['GM'] / d['REVENUE'] * 100).round(1))
    .reset_index()
)

# Plot by region for current year
gm_cy = gm_region[gm_region['YEAR'] == CURRENT_YEAR]

fig, ax = plt.subplots(figsize=(12, 5))
for region, grp in gm_cy.groupby('REGION'):
    ax.plot(grp['MONTH'], grp['GM_PCT'], marker='o', linewidth=2, label=region)

ax.set_title(f'Gross Margin % by Region — {CURRENT_YEAR}', fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('GM %')
ax.set_xticks(range(1, 13))
ax.axhline(gm_cy['GM_PCT'].mean(), color='black', linestyle=':', alpha=0.5, label='Avg')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.savefig('day25_gm_trend.png', dpi=120, bbox_inches='tight')
plt.show()

## 3. Top 10 Customers by Revenue + Retention Flag

In [ ]:
# YOUR SOLUTION
# Customers present in BOTH CURRENT_YEAR and PRIOR_YEAR = retained
cust_cy = set(bw[bw['YEAR'] == CURRENT_YEAR]['KUNNR'].unique())
cust_py = set(bw[bw['YEAR'] == PRIOR_YEAR]['KUNNR'].unique())
retained = cust_cy & cust_py

cust_rev = (
    bw[bw['YEAR'] == CURRENT_YEAR]
    .groupby('KUNNR')['REVENUE'].sum()
    .reset_index()
    .sort_values('REVENUE', ascending=False)
    .head(10)
)
cust_rev['RETAINED'] = cust_rev['KUNNR'].isin(retained)
cust_rev['RETENTION_FLAG'] = cust_rev['RETAINED'].map({True: 'Retained', False: 'New'})

print('=== Top 10 Customers by Revenue ===')
print(cust_rev[['KUNNR','REVENUE','RETENTION_FLAG']].to_string(index=False))

## 4. Returns Rate by Product Division

In [ ]:
# YOUR SOLUTION
returns_div = (
    bw.groupby('SPART')
    .agg(REVENUE=('REVENUE','sum'), RETURNS_VALUE=('RETURNS_VALUE','sum'))
    .assign(RETURNS_RATE=lambda d: (d['RETURNS_VALUE'] / d['REVENUE'] * 100).round(2))
    .reset_index()
    .sort_values('RETURNS_RATE', ascending=False)
)
returns_div.columns = ['Division','Revenue','Returns Value','Returns Rate %']
print('=== Returns Rate by Division ===')
print(returns_div.to_string(index=False))

## 5. Discount Impact Analysis

In [ ]:
# YOUR SOLUTION
disc = (
    bw.groupby(['YEAR', 'REGION'])
    .agg(REVENUE=('REVENUE','sum'), DISCOUNT=('DISCOUNT_AMT','sum'))
    .assign(DISC_PCT=lambda d: (d['DISCOUNT'] / d['REVENUE'] * 100).round(1))
    .reset_index()
)

print('=== Discount Impact by Year and Region ===')
print(disc.to_string(index=False))

# What revenue would look like without discounts
bw['GROSS_REVENUE'] = bw['REVENUE'] + bw['DISCOUNT_AMT']
total_lost_to_discount = bw['DISCOUNT_AMT'].sum()
print(f'\nTotal revenue impact of discounts: ${total_lost_to_discount:,.0f}')

## 6. KPI Cards (Text Output) + Remaining Charts

In [ ]:
# KPI CARDS — print a management summary
gm_cy_total = bw[bw['YEAR'] == CURRENT_YEAR]['GROSS_MARGIN'].sum()
rev_cy_total = bw[bw['YEAR'] == CURRENT_YEAR]['REVENUE'].sum()
gm_pct_cy = gm_cy_total / rev_cy_total * 100

returns_total_pct = (bw[bw['YEAR']==CURRENT_YEAR]['RETURNS_VALUE'].sum() /
                     bw[bw['YEAR']==CURRENT_YEAR]['REVENUE'].sum() * 100)

disc_total_pct = (bw[bw['YEAR']==CURRENT_YEAR]['DISCOUNT_AMT'].sum() /
                  bw[bw['YEAR']==CURRENT_YEAR]['REVENUE'].sum() * 100)

retention_rate = len(retained) / len(cust_py) * 100 if cust_py else 0

cards = [
    ('Revenue (CY)', f'${rev_cy_total:,.0f}', f'YoY: {yoy_total_pct:+.1f}%'),
    ('Gross Margin %', f'{gm_pct_cy:.1f}%', 'CY full year'),
    ('Returns Rate', f'{returns_total_pct:.2f}%', 'of Revenue'),
    ('Discount Rate', f'{disc_total_pct:.1f}%', 'of Revenue'),
    ('Customer Retention', f'{retention_rate:.0f}%', 'CY vs PY customers'),
]

print('=' * 55)
print(' ' * 15 + 'MANAGEMENT KPI DASHBOARD')
print('=' * 55)
for name, value, note in cards:
    print(f'  {name:<25} {value:<15} {note}')
print('=' * 55)

# Chart 2: YoY Revenue Bars
fig, ax = plt.subplots(figsize=(11, 4))
x = yoy['MONTH']
width = 0.35
ax.bar(x - width/2, yoy['CY_REVENUE']/1e6, width, label=str(CURRENT_YEAR), color='#3498db')
ax.bar(x + width/2, yoy['PY_REVENUE']/1e6, width, label=str(PRIOR_YEAR), color='#95a5a6')
ax.set_title('Monthly Revenue: Current Year vs Prior Year', fontweight='bold')
ax.set_ylabel('Revenue ($M)')
ax.set_xticks(range(1,13))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.1f}M'))
ax.legend()
plt.tight_layout()
plt.savefig('day25_yoy_revenue.png', dpi=120, bbox_inches='tight')
plt.show()

# Chart 3: Returns rate by division
fig, ax = plt.subplots(figsize=(8, 4))
returns_div.plot(kind='bar', x='Division', y='Returns Rate %', ax=ax,
                 color='#e74c3c', legend=False, edgecolor='white')
ax.set_title('Returns Rate by Product Division', fontweight='bold')
ax.set_ylabel('Returns Rate (%)')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig('day25_returns_rate.png', dpi=120, bbox_inches='tight')
plt.show()

## Daily Prompt
**Answer independently:**

1. A stakeholder asks why Q3 gross margin dropped 3 points vs prior year. Using only the BW extract, identify which region and which product division drove the decline. Write the pandas code to answer this.
2. In SAP BW, what is the difference between an InfoCube and a DSO (DataStore Object)? Which one would you use as the source for this KPI extract, and why?
3. The retention flag currently only checks if a customer appeared in both years. Enhance it to show: customers where CY revenue is ≥80% of PY revenue (growing/stable retained), ≥1% but <80% (declining retained), and new (no PY appearance).
